### VPC Flow Log Hunting: part of the DUNE project
Ingests VPC flow logs, labels known cloud provider IPs, tor IPs, and anything known you want to define in known_ips.txt. Flows are then aggregated into records with summed time spans, byte and packet counts. Flows are tagged north-south or east-west (tag-direction) and ingress or egress (flow-direction). At the end of the notebook below, three interfaces exist for hunting:
1. Flows counted by total bytes or time duration
2. Flows counted by time duration (for comparison)
3. Beaconing candidates - flows with regular frequency and byte counts
   
Field definitions:

tag-direction: north-south (involving external networks) or east-west (staying within private networks).

flow-direction: ingress or egress, relative to the ENI, from the v5 flow log field. Older (v2/v3) logs don't carry it and it isn't derived, so the column and the direction filters are absent for them.

action: accept or reject, according to the flow records

tag-org: the organisation the flow's public end belongs to -- cloud provider / service (AWS, Azure, Google, Akamai, Cloudflare, Fastly, Zscaler, Tor) or your own known_ips.txt name

tag-service: service name from conf/tcp-ports.md / udp-ports.md for the session's listed port; "anomalous" when src and dst ports name two different services (scan traffic from a fixed well-known port)

tag-label: behaviour tags, e.g. "beaconing" (cell 19)


Fields the notebook derives are prefixed `tag-` so they never collide with flow log fields. The flow log's own fields, such as `type` (IPv4 / IPv6 / EFA), are kept as-is.


In [ ]:
# --- cell 00 ---
import pandas as pd
import ipaddress
from ipaddress import ip_address
import numpy as np
import os
import json
import re
from functools import reduce
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# plotly 6 FigureWidget: after a redraw the front end can report a trace delta with
# no uid, and plotly's handler dies with KeyError: 'uid'. Fill the uid in by trace
# position before plotly reads it. Guarded so re-running this cell doesn't double-wrap.

from plotly.basewidget import BaseFigureWidget
if not getattr(BaseFigureWidget, "_uid_patch", False):
    _orig_trace_deltas = BaseFigureWidget._handler_js2py_traceDeltas.func
    def _trace_deltas_with_uid(self, change):
        for i, d in enumerate((change["new"] or {}).get("trace_deltas") or []):
            if "uid" not in d and i < len(self.data):
                d["uid"] = self.data[i].uid
        return _orig_trace_deltas(self, change)
    BaseFigureWidget._handler_js2py_traceDeltas = _trace_deltas_with_uid
    BaseFigureWidget._uid_patch = True

In [ ]:
# --- cell 01 ---
# All reference CIDR data lives here (local snapshots, no network needed)

CIDR_DIR   = "cidrs"
_REF_COLS  = ["ip_prefix", "region", "service", "network_border_group"]

# Load AWS IP ranges from the local snapshot in cidrs/.
# Missing / unreadable file -> warn and carry on with an empty frame.

try:
    with open(os.path.join(CIDR_DIR, "ip-ranges.json")) as _f:
        aws_data = json.load(_f)
    aws = pd.DataFrame(aws_data["prefixes"])[_REF_COLS]
except (OSError, ValueError, KeyError) as _e:
    print(f"!! AWS ip-ranges.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    aws = pd.DataFrame(columns=_REF_COLS)

aws

In [ ]:
# --- cell 02 ---
# Load the already-normalized Zscaler CIDR list from cidrs/zscaler.csv.
# Schema matches `aws`: ip_prefix, region, service, network_border_group, network.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    zscaler = pd.read_csv(os.path.join(CIDR_DIR, "zscaler.csv"))
    zscaler["network"] = zscaler["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
except (OSError, ValueError, KeyError) as _e:
    print(f"!! zscaler.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    zscaler = pd.DataFrame(columns=_SCHEMA)

print(f"{len(zscaler)} Zscaler CIDRs")
zscaler

In [ ]:
# --- cell 03 ---
# Load the already-normalized Tor exit list from cidrs/tor.csv.
# Schema matches `aws`; each entry is a /32 host. service = "TOR".
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    tor = pd.read_csv(os.path.join(CIDR_DIR, "tor.csv"))
    tor["network"] = tor["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
except (OSError, ValueError, KeyError) as _e:
    print(f"!! tor.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    tor = pd.DataFrame(columns=_SCHEMA)

print(f"{len(tor)} Tor exit IPs")
tor

In [ ]:
# --- cell 04 ---
# Azure Service Tags (cidrs/azure.json) -> same schema as `aws`.
# Structure differs from AWS: values[] = service tags, CIDRs nested in
# properties.addressPrefixes[] -> explode. service = systemService, falling back
# to the tag `name` when systemService is blank. IPv6 dropped. One row per CIDR:
# a specific tag beats the AzureCloud umbrella, and a tag with a region beats one without.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    with open(os.path.join(CIDR_DIR, "azure.json")) as _f:
        _az = json.load(_f)

    _rows = []
    for _v in _az["values"]:
        _p   = _v["properties"]
        _svc = _p.get("systemService") or _v["name"]
        _reg = _p.get("region") or ""
        for _pfx in _p.get("addressPrefixes", []):
            if ":" in _pfx:                       # IPv4 only
                continue
            _rows.append({"ip_prefix": _pfx, "region": _reg, "service": _svc,
                          "_umbrella": _v["name"].startswith("AzureCloud")})

    azure = (pd.DataFrame(_rows)
             .sort_values(["ip_prefix", "_umbrella", "region"],
                          ascending=[True, True, False], kind="stable")
             .drop_duplicates("ip_prefix", keep="first")
             .drop(columns="_umbrella")
             .reset_index(drop=True))
    azure["network_border_group"] = pd.NA
    azure["region"]  = azure["region"].replace("", pd.NA)
    azure["network"] = azure["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    azure = azure[_SCHEMA]
    _chg = _az.get("changeNumber")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! azure.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    azure, _chg = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(azure)} Azure CIDRs  ({azure['service'].nunique()} services, changeNumber {_chg})")
azure

In [ ]:
# --- cell 05 ---
# Google ranges: goog.json (all Google) + gcloud.json (GCP customer ranges, a
# subset of goog) -> same schema as `aws`. IPv4 only. service/region come from
# gcloud; the goog-only remainder is a generic "GOOGLE". A specific gcloud prefix
# beats a broad goog block via most-specific-match downstream; an exact-duplicate
# prefix keeps the gcloud row (it carries service + region).
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
try:
    with open(os.path.join(CIDR_DIR, "gcloud.json")) as _f:
        _gc = json.load(_f)
    with open(os.path.join(CIDR_DIR, "goog.json")) as _f:
        _gg = json.load(_f)

    _rows = [{"ip_prefix": _p["ipv4Prefix"], "region": _p.get("scope"),
              "service": _p.get("service") or "Google Cloud", "_src": "gcloud"}
             for _p in _gc["prefixes"] if "ipv4Prefix" in _p]
    _rows += [{"ip_prefix": _p["ipv4Prefix"], "region": pd.NA,
               "service": "GOOGLE", "_src": "goog"}
              for _p in _gg["prefixes"] if "ipv4Prefix" in _p]

    google = (pd.DataFrame(_rows)
              .sort_values("_src", kind="stable")          # gcloud rows before goog
              .drop_duplicates("ip_prefix", keep="first")
              .drop(columns="_src")
              .reset_index(drop=True))
    google["network_border_group"] = pd.NA
    google["region"]  = google["region"].replace("", pd.NA)
    google["network"] = google["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    google = google[_SCHEMA]
    _sync = _gc.get("syncToken")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! Google ranges not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    google, _sync = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(google)} Google CIDRs  "
      f"({int((google['service'] == 'Google Cloud').sum())} GCP, "
      f"{int((google['service'] == 'GOOGLE').sum())} other Google, syncToken {_sync})")
google

In [ ]:
# --- cell 05a ---
# Akamai prefixes from cidrs/akamai.csv (IPv4 announced by Akamai's ASNs, via RIPEstat;
# refresh with `python cidrs/fetch_akamai.py`). Schema matches `aws`. service is one of
# AKAMAI (CDN / edge), AKAMAI-PROLEXIC (DDoS scrubbing, can include customer space) or
# AKAMAI-LINODE (rentable cloud VMs -- NOT a trust signal, anyone can host there).
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
_AK_FILE = os.path.join(CIDR_DIR, "akamai.csv")
try:
    akamai = pd.read_csv(_AK_FILE)
    akamai["network"] = akamai["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    akamai = akamai[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_AK_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! akamai.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    akamai, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(akamai)} Akamai CIDRs" + (f"  (file {_age} days old)" if _age is not None else ""))
print(akamai["service"].value_counts().to_string())
akamai

In [ ]:
# --- cell 05b ---
# Cloudflare's official proxy / CDN edge ranges from cidrs/cloudflare.csv (refresh with
# `python cidrs/fetch_cloudflare.py`). Schema matches `aws`; service = CLOUDFLARE.
# Official list only -- 1.1.1.1 DNS, WARP egress and customer BYOIP space are not in it.
# Anyone can front a site with Cloudflare (Workers, Tunnels, R2), so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
_CF_FILE = os.path.join(CIDR_DIR, "cloudflare.csv")
try:
    cloudflare = pd.read_csv(_CF_FILE)
    cloudflare["network"] = cloudflare["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    cloudflare = cloudflare[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_CF_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! cloudflare.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    cloudflare, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(cloudflare)} Cloudflare CIDRs" + (f"  (file {_age} days old)" if _age is not None else ""))
cloudflare

In [ ]:
# --- cell 05c ---
# Port -> service names from conf/tcp-ports.md (protocol 6) and conf/udp-ports.md (17).
# Only lines starting with a port number are read: "port<TAB>service<TAB>description",
# where "67 / 68" lists several ports sharing one name. Headings and column-header
# lines are skipped. A port listed twice in one file keeps its first name (warned).
# Missing / unreadable file -> warn and carry on without that protocol's names.
# Also IP protocol number -> name from conf/protocols.md ("<number> <name>", any spacing;
# everything after the number is the name) -> `protocolNames`, used by cell 14a for
# sessions that aren't TCP / UDP.

CONF_DIR    = "conf"
_PORT_FILES = {6: "tcp-ports.md", 17: "udp-ports.md"}

_rows = []
for _proto, _fn in _PORT_FILES.items():
    try:
        with open(os.path.join(CONF_DIR, _fn), encoding="utf-8") as _f:
            for _no, _ln in enumerate(_f, 1):
                if not _ln.strip()[:1].isdigit():
                    continue
                _parts = [p.strip() for p in re.split(r"\t+|\s{2,}", _ln.strip())]
                try:
                    _ports = [int(p) for p in _parts[0].split("/")]
                    if len(_parts) < 2 or not _parts[1] or not all(0 <= p <= 65535 for p in _ports):
                        raise ValueError
                except ValueError:
                    print(f"!! {_fn} line {_no} not understood, skipped: {_ln.strip()!r}")
                    continue
                _rows += [{"protocol": _proto, "port": p, "service": _parts[1]} for p in _ports]
    except OSError as _e:
        print(f"!! {_fn} not loaded ({type(_e).__name__}: {_e}) -> no protocol-{_proto} service names")

portServices = pd.DataFrame(_rows, columns=["protocol", "port", "service"])
_dupe = portServices.duplicated(["protocol", "port"], keep="first")
for _, _r in portServices[_dupe].iterrows():
    print(f"!! protocol {_r.protocol} port {_r.port} listed again as {_r.service!r} -- keeping the first name")
portServices = portServices[~_dupe].reset_index(drop=True)

protocolNames = {}
try:
    with open(os.path.join(CONF_DIR, "protocols.md"), encoding="utf-8") as _f:
        for _no, _ln in enumerate(_f, 1):
            _m = re.match(r"\s*(\d+)\s+(.+?)\s*$", _ln)
            if not _ln.strip()[:1].isdigit():
                continue
            if not _m or not 0 <= int(_m.group(1)) <= 255:
                print(f"!! protocols.md line {_no} not understood, skipped: {_ln.strip()!r}")
                continue
            if int(_m.group(1)) in protocolNames:
                print(f"!! protocols.md: protocol {_m.group(1)} listed again -- keeping the first name")
                continue
            protocolNames[int(_m.group(1))] = _m.group(2)
except OSError as _e:
    print(f"!! protocols.md not loaded ({type(_e).__name__}: {_e}) -> non-TCP/UDP sessions get no tag-service")

print(f"{len(protocolNames)} protocol names: " + ", ".join(f"{k}={v}" for k, v in sorted(protocolNames.items())))
print(f"{len(portServices)} port names  "
      + ", ".join(f"{_PORT_FILES[k]}: {v}" for k, v in portServices["protocol"].value_counts().sort_index().items()))
portServices

In [ ]:
# --- cell 05d ---
# Fastly's official CDN / edge ranges from cidrs/fastly.csv (refresh with
# `python cidrs/fetch_fastly.py`). Schema matches `aws`; service = FASTLY.
# Anyone can serve content or run code (Compute@Edge) via Fastly, so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group", "network"]
_FL_FILE = os.path.join(CIDR_DIR, "fastly.csv")
try:
    fastly = pd.read_csv(_FL_FILE)
    fastly["network"] = fastly["ip_prefix"].apply(lambda c: ipaddress.ip_network(c, strict=False))
    fastly = fastly[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_FL_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! fastly.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    fastly, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(fastly)} Fastly CIDRs" + (f"  (file {_age} days old)" if _age is not None else ""))
fastly

In [ ]:
# --- cell 06 ---
# load flow data
# PARQUET_PATH is either one .parquet file or a folder -- a folder loads every
# *.parquet directly inside it (not subfolders) and stacks them. Files are read one
# at a time so exports with different column types still combine; an unreadable
# file in a folder is skipped with a warning.
# start/end must be epoch seconds (Int64) -- cell 08's MERGE_GAP and the duration /
# beacon cells do plain-second arithmetic. Some exports store them as timestamps
# or durations instead; normalise those here so every cell sees the same type.
# An export may carry start/end OR start_time/end_time; the missing pair is derived
# from the other, so vflows always has all four.

PARQUET_PATH = "vflows.parquet"

def _norm_times(df):
    """start/end -> epoch seconds (Int64); start_time/end_time -> datetime64[ms, UTC].
    A file may carry only one of the pairs -- the other is derived from it."""
    for _a, _t in (("start", "start_time"), ("end", "end_time")):
        if _a in df.columns:
            _s = df[_a]
            if pd.api.types.is_datetime64_any_dtype(_s):
                _s = _s.dt.tz_localize("UTC") if _s.dt.tz is None else _s
                df[_a] = ((_s - pd.Timestamp(0, tz="UTC")) // pd.Timedelta(seconds=1)).astype("Int64")
            elif pd.api.types.is_timedelta64_dtype(_s):
                df[_a] = (_s // pd.Timedelta(seconds=1)).astype("Int64")
            else:
                df[_a] = pd.to_numeric(_s, errors="coerce").astype("Int64")
        if _t in df.columns:
            df[_t] = pd.to_datetime(df[_t], utc=True, errors="coerce").astype("datetime64[ms, UTC]")
        if _a in df.columns and _t not in df.columns:
            df[_t] = pd.to_datetime(df[_a].astype("float64"), unit="s", utc=True).astype("datetime64[ms, UTC]")
        elif _t in df.columns and _a not in df.columns:
            df[_a] = ((df[_t] - pd.Timestamp(0, tz="UTC")) // pd.Timedelta(seconds=1)).astype("Int64")
    return df

if os.path.isdir(PARQUET_PATH):
    _files = sorted(os.path.join(PARQUET_PATH, f) for f in os.listdir(PARQUET_PATH)
                    if f.lower().endswith(".parquet"))
    if not _files:
        raise FileNotFoundError(f"no .parquet files in folder {PARQUET_PATH!r}")
elif os.path.isfile(PARQUET_PATH):
    _files = [PARQUET_PATH]
else:
    raise FileNotFoundError(f"PARQUET_PATH {PARQUET_PATH!r} is not a file or folder")

_frames = []
for _f in _files:
    try:
        _frames.append(_norm_times(pd.read_parquet(_f)))
    except Exception as _e:
        if len(_files) == 1:
            raise
        print(f"!! skipped {os.path.basename(_f)} ({type(_e).__name__}: {_e})")
if not _frames:
    raise ValueError(f"none of the {len(_files)} parquet files in {PARQUET_PATH!r} could be read")

_nRead = len(_frames)
vflows = pd.concat(_frames, ignore_index=True) if _nRead > 1 else _frames[0]
del _frames

print(f"loaded {len(vflows):,} rows from {PARQUET_PATH!r}"
      + (f" ({_nRead} of {len(_files)} files)" if os.path.isdir(PARQUET_PATH) else ""))
_tc = [c for c in ("start", "end", "start_time", "end_time") if c in vflows.columns]
print(vflows[_tc].dtypes.to_string())
vflows.shape

In [ ]:
# --- cell 07 ---
# Duplicate rows in vflows, matched on DUP_COLS only. Comment a column out to
# ignore it, e.g. drop interface-id to catch the same flow logged by two ENIs.

DUP_COLS = [
    "version",
    "account-id",
    #"interface-id",
    "srcaddr",
    "dstaddr",
    "srcport",
    "dstport",
    "protocol",
    #"packets",
    #"bytes",
    #"start",
    #"end",
    # "start_time",    # same instant as start
    # "end_time",      # same instant as end
    "action",
    "log-status",
]

_missing = [c for c in DUP_COLS if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")

_dup   = vflows.duplicated(subset=DUP_COLS, keep=False)
_extra = int(vflows.duplicated(subset=DUP_COLS, keep="first").sum())

print(f"matching on {len(DUP_COLS)} columns: {', '.join(DUP_COLS)}")
print(f"{len(vflows):,} rows")
print(f"{_extra:,} extra copies (rows you'd drop to dedupe)  {_extra / len(vflows):.2%}")
print(f"{int(_dup.sum()):,} rows involved in duplication, "
      f"{int(_dup.sum()) - _extra:,} distinct duplicated records")

vflowsDups = (vflows[_dup].groupby(DUP_COLS, dropna=False).size()
               .rename("copies").sort_values(ascending=False).reset_index())
vflowsDups.head(20)

In [ ]:
# --- cell 08 ---
# Aggregate repeated flow records into sessions -> vflowsAgg (vflows untouched).
# Key = DUP_COLS from the cell above, minus the per-record counters and times.
# Records with the same key merge while each starts within MERGE_GAP seconds of
# the latest end so far; a longer gap starts a new session, so repeat connections
# (scans, beacons, ICMP, reused source ports) keep their timing.
# NODATA (no traffic in the window) and SKIPDATA (records lost at capture) rows carry
# only ENI + time -- set the flags to leave them out of vflowsAgg.
# Fails fast with a pointer to the cell to fix if its inputs aren't what it expects.

MERGE_GAP     = 600     # seconds -- AWS's longest flow-log aggregation window
DROP_NODATA   = False
DROP_SKIPDATA = False

_per_rec = ["packets", "bytes", "start", "end", "start_time", "end_time"]

if "DUP_COLS" not in globals():
    raise NameError("DUP_COLS not defined -- run cell 07 first")
_missing = [c for c in dict.fromkeys(DUP_COLS + _per_rec) if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")
if vflows.empty:
    raise ValueError("vflows is empty -- nothing to aggregate (check PARQUET_PATH in cell 06)")
_bad = [c for c in ("start", "end", "packets", "bytes")
        if not pd.api.types.is_numeric_dtype(vflows[c]) or pd.api.types.is_bool_dtype(vflows[c])]
if _bad:
    raise TypeError("must be numeric (start/end in epoch seconds): "
                    + ", ".join(f"{c} is {vflows[c].dtype}" for c in _bad) + " -- re-run cell 06")

_v    = vflows[[c for c in vflows.columns if not c.startswith("tag-")]]   # tag-* fields are derived later; log fields (e.g. `type`) are kept
_drop = [s for s, f in (("NODATA", DROP_NODATA), ("SKIPDATA", DROP_SKIPDATA)) if f]
if _drop and "log-status" not in _v.columns:
    print(f"!! no log-status column -- can't drop {', '.join(_drop)}")
elif _drop:
    _m = _v["log-status"].isin(_drop)
    print(f"dropped {int(_m.sum()):,} {'/'.join(_drop)} records (vflows keeps them)")
    _v = _v[~_m]
if _v.empty:
    raise ValueError("no records left to aggregate after dropping " + "/".join(_drop))

_key = [c for c in DUP_COLS if c not in _per_rec]
_noTime = int(_v["start"].isna().sum() + _v["end"].isna().sum())
if _noTime:
    print(f"!! {_noTime:,} missing start/end values -- those records become sessions of their own")

_gid = _v.groupby(_key, dropna=False, sort=False).ngroup()
_v   = _v.assign(_gid=_gid).sort_values(["_gid", "start"], kind="stable")

_prev_end = _v.groupby("_gid")["end"].cummax().groupby(_v["_gid"]).shift()
_new      = (_v["_gid"].ne(_v["_gid"].shift())
             | (_v["start"] - _prev_end).gt(MERGE_GAP).fillna(_v["start"].isna()))
_v["_sid"] = _new.astype(bool).cumsum()

_aggs = {c: (c, "first") for c in _v.columns if c not in _per_rec + ["_gid", "_sid"]}
_aggs.update(start=("start", "min"), end=("end", "max"),
             start_time=("start_time", "min"), end_time=("end_time", "max"),
             records=("start", "size"))

_g   = _v.groupby("_sid", sort=False)
_out = _g.agg(**_aggs)
_out[["packets", "bytes"]] = _g[["packets", "bytes"]].sum(min_count=1)   # NODATA stays <NA>, not 0

vflowsAgg = (_out[[c for c in vflows.columns if c in _out.columns] + ["records"]]
              .sort_values("start", kind="stable")
              .reset_index(drop=True))

if int(vflowsAgg["records"].sum()) != len(_v):
    raise AssertionError(f"record count mismatch: {len(_v):,} in, "
                         f"{int(vflowsAgg['records'].sum()):,} accounted for in vflowsAgg")

_merged = int((vflowsAgg["records"] > 1).sum())
print(f"key ({len(_key)}): {', '.join(_key)}   gap <= {MERGE_GAP}s")
print(f"{len(_v):,} records -> {len(vflowsAgg):,} sessions  "
      f"({len(_v) - len(vflowsAgg):,} folded in, {_merged:,} sessions built from >1 record)")
print(f"bytes   {_v['bytes'].sum():,} -> {vflowsAgg['bytes'].sum():,}")
print(f"packets {_v['packets'].sum():,} -> {vflowsAgg['packets'].sum():,}")
vflowsAgg.sort_values("records", ascending=False).head(10)

In [ ]:
# --- cell 09 ---
# unique public IPs across ALL flows (either end). Vectorised end to end:
# parse every distinct address once, classify by integer range (no per-IP Python
# / ipaddress calls), and keep the int for the CIDR match downstream.

def ipv4_to_int(s):
    o = pd.Series(s, dtype="string").str.split(".", expand=True).astype("uint64")
    return (o[0] << np.uint64(24) | o[1] << np.uint64(16)
            | o[2] << np.uint64(8) | o[3]).to_numpy("uint64")

# IPv4 blocks ipaddress marks non-global (RFC1918, CGNAT, loopback, link-local,
# doc/benchmark ranges, multicast + reserved + broadcast)

_NON_GLOBAL = [
    ("0.0.0.0", "0.255.255.255"),       ("10.0.0.0", "10.255.255.255"),
    ("100.64.0.0", "100.127.255.255"),  ("127.0.0.0", "127.255.255.255"),
    ("169.254.0.0", "169.254.255.255"), ("172.16.0.0", "172.31.255.255"),
    ("192.0.0.0", "192.0.0.255"),       ("192.0.2.0", "192.0.2.255"),
    ("192.88.99.0", "192.88.99.255"),   ("192.168.0.0", "192.168.255.255"),
    ("198.18.0.0", "198.19.255.255"),   ("198.51.100.0", "198.51.100.255"),
    ("203.0.113.0", "203.0.113.255"),   ("224.0.0.0", "255.255.255.255"),
]

_ips  = pd.unique(np.concatenate([vflowsAgg["srcaddr"].unique(), vflowsAgg["dstaddr"].unique()]))
_ips  = _ips[pd.Series(_ips).str.count(r"\.").eq(3).to_numpy()]     # drop None / IPv6
_ints = ipv4_to_int(_ips)

_pub = np.ones(len(_ips), dtype=bool)
for _lo, _hi in _NON_GLOBAL:
    _pub &= ~((_ints >= ipv4_to_int([_lo])[0]) & (_ints <= ipv4_to_int([_hi])[0]))

_ord   = np.argsort(_ints[_pub], kind="stable")
public = pd.DataFrame({"ip": _ips[_pub][_ord], "ip_int": _ints[_pub][_ord]})
print(f"{len(public)} unique public IPs  (of {len(_ips)} distinct IPv4 across all flows)")
print(public.isna().sum())
public

In [ ]:
# --- cell 10 ---
# Traffic direction per session in vflowsAgg -> `tag-direction`: north-south or east-west.
# Reuses `public` from the cell above, so public/private has one definition. Either end
# public -> north-south; both ends valid IPv4 and non-public -> east-west. Address-less /
# IPv6 rows stay <NA>. (Ingress / egress is the separate `flow-direction`, cell 11.)

_pub_ips = set(public["ip"])
_src_pub = vflowsAgg["srcaddr"].isin(_pub_ips)
_dst_pub = vflowsAgg["dstaddr"].isin(_pub_ips)
_src_v4  = vflowsAgg["srcaddr"].astype("string").str.count(r"\.").eq(3).fillna(False)
_dst_v4  = vflowsAgg["dstaddr"].astype("string").str.count(r"\.").eq(3).fillna(False)

vflowsAgg["tag-direction"] = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
vflowsAgg.loc[_src_v4 & _dst_v4 & ~_src_pub & ~_dst_pub, "tag-direction"] = "east-west"
vflowsAgg.loc[_src_pub | _dst_pub, "tag-direction"] = "north-south"

print(vflowsAgg["tag-direction"].value_counts(dropna=False).to_string())

In [ ]:
# --- cell 11 ---
# Ingress / egress per session in vflowsAgg -> `flow-direction`, the v5 flow log field
# (ENI-relative). If the logs carry it, it's cleaned up in place (lower case, "-" -> <NA>);
# an Athena / Glue `flow_direction` column is renamed to `flow-direction` first.
# v2/v3 logs don't have it, and it is NOT derived -- guessing from which end is public
# isn't reliable -- so those runs have no flow-direction column, and the direction
# filters below are greyed out.

if "flow-direction" not in vflowsAgg.columns and "flow_direction" in vflowsAgg.columns:
    vflowsAgg = vflowsAgg.rename(columns={"flow_direction": "flow-direction"})
    print("flow-direction: renamed from the export's flow_direction column")

if "flow-direction" in vflowsAgg.columns:
    vflowsAgg["flow-direction"] = (vflowsAgg["flow-direction"].astype("string")
                                   .str.strip().str.lower().replace("-", pd.NA))
    print("flow-direction: taken from the log")
    print(vflowsAgg["flow-direction"].value_counts(dropna=False).to_string())
else:
    _v = pd.to_numeric(vflowsAgg["version"], errors="coerce").max()
    print(f"!! no flow-direction field (v{int(_v)} logs) -- not derived; direction filters disabled")


In [ ]:
# --- cell 12 ---
# One interval index over AWS + Zscaler + Tor + Azure + Google + Akamai + Cloudflare + Fastly (schema-identical frames).
# Vectorised CIDR -> [start, end]; one binary search per IP; most-specific prefix
# wins, so a Zscaler/Azure /24 beats a broad AWS /12. ~0.3 s for 53k prefixes.
# A reference frame that's undefined, empty or missing columns is reported and left
# out; labelling carries on with the rest (or with none, leaving every tag-org <NA>).
# Writes the matched org / service name to public["tag-org"].

_REF_NAMES = ["aws", "zscaler", "tor", "azure", "google", "akamai", "cloudflare", "fastly"]
_REF_NEED  = ["ip_prefix", "service", "region"]

_refOk = []
for _n in _REF_NAMES:
    _df = globals().get(_n)
    if not isinstance(_df, pd.DataFrame):
        print(f"!! {_n} not defined -- run its load cell; no {_n} labels")
    elif _df.empty:
        print(f"!! {_n} is empty -- no {_n} labels")
    elif _miss := [c for c in _REF_NEED if c not in _df.columns]:
        print(f"!! {_n} is missing columns {_miss} -- no {_n} labels")
    else:
        _refOk.append(_n)

if "aws" in _refOk and not (aws["ip_prefix"] == "72.21.192.0/19").any():
    aws = pd.concat([aws, pd.DataFrame([{
        "ip_prefix": "72.21.192.0/19", "region": "-",
        "service": "AMAZON", "network_border_group": "-"}])], ignore_index=True)

refs = (pd.concat([globals()[_n].assign(ref=_n) for _n in _refOk], ignore_index=True)[_REF_NEED + ["ref"]]
        if _refOk else pd.DataFrame(columns=_REF_NEED + ["ref"]))

def cidr_bounds(prefixes):
    """Vectorised 'a.b.c.d/n' -> (start_int, end_int) arrays, uint32 range in uint64."""
    p    = pd.Series(prefixes, dtype="string").str.split("/", n=1, expand=True)
    o    = p[0].str.split(".", expand=True).astype("uint64")
    base = (o[0] << np.uint64(24) | o[1] << np.uint64(16)
            | o[2] << np.uint64(8) | o[3]).to_numpy("uint64")
    plen = p[1].fillna("32").astype("uint64").to_numpy("uint64")
    mask = np.where(
        plen == 0, np.uint64(0),
        (np.uint64(0xFFFFFFFF) << (np.uint64(32) - plen)) & np.uint64(0xFFFFFFFF),
    ).astype("uint64")
    start = base & mask
    return start, start | (~mask & np.uint64(0xFFFFFFFF))

def match_cidr_index(idx, ip_ints):
    """Per IP: row index of the most specific containing prefix, or -1."""
    ip_ints = np.asarray(ip_ints, dtype="uint64")
    starts, ends, cummax = idx["starts"], idx["ends"], idx["cummax"]
    n    = len(starts)
    out  = np.full(len(ip_ints), -1, dtype=np.int64)
    if n == 0:                                                      # no reference data
        return out
    pos  = np.searchsorted(starts, ip_ints, side="right") - 1
    posc = np.clip(pos, 0, n - 1)
    direct  = (pos >= 0) & (ends[posc]   >= ip_ints)
    covered = (pos >= 0) & (cummax[posc] >= ip_ints)
    out[direct] = pos[direct]
    for k in np.where(covered & ~direct)[0]:                        # rare: nested broad block
        j, ipv = int(pos[k]), ip_ints[k]
        while j >= 0 and ends[j] < ipv:
            j -= 1
        out[k] = j
    return out

if len(refs):
    _st, _en = cidr_bounds(refs["ip_prefix"])
else:
    _st = _en = np.empty(0, dtype="uint64")

# Sort by start asc, then end desc. For same-start overlaps, searchsorted(side="right")
# lands on the shortest containing prefix, i.e. the most-specific CIDR.

_o = np.lexsort((np.uint64(0xFFFFFFFF) - _en, _st))
ref_idx = {"starts": _st[_o], "ends": _en[_o],
           "cummax": np.maximum.accumulate(_en[_o]) if len(_o) else _en[_o],
           "rows": refs.iloc[_o].reset_index(drop=True)}

public["ref_row"] = match_cidr_index(ref_idx, public["ip_int"].to_numpy("uint64"))
public["in_ref"]  = public["ref_row"] >= 0
_rr  = public["ref_row"].clip(lower=0).to_numpy()
_hit = public["in_ref"].to_numpy()
_rows = ref_idx["rows"] if len(refs) else pd.DataFrame({c: [np.nan] for c in _REF_NEED + ["ref"]})
public["tag-org"]      = np.where(_hit, _rows["service"].to_numpy()[_rr],   np.nan)
public["tag-org-source"] = np.where(_hit, _rows["ref"].to_numpy()[_rr],       np.nan)
public["ref_prefix"]   = np.where(_hit, _rows["ip_prefix"].to_numpy()[_rr], np.nan)
public["aws_region"]   = np.where(_hit & (public["tag-org-source"].to_numpy() == "aws"),
                                  _rows["region"].to_numpy()[_rr], np.nan)
_lblCols = ["tag-org", "tag-org-source", "ref_prefix", "aws_region"]
public[_lblCols] = public[_lblCols].astype(object)    # stays text-friendly even with no matches

print(f"reference sets used: {', '.join(_refOk) or 'NONE'}  ({len(refs):,} prefixes)")
print(public.loc[public["in_ref"], "tag-org-source"].value_counts().to_string())
print(f"matched {int(public['in_ref'].sum())} / {len(public)} public IPs")
public.loc[public["in_ref"], ["ip", "ref_prefix", "tag-org", "tag-org-source"]].head()

In [ ]:
# --- cell 13 ---
# Known / trusted addresses from known_ips.txt (kept out of the notebook so it
# never gets committed). One IP or CIDR per line, optional label, # comments.
# Tags `public` with known / known_label, and writes known_label into `tag-org`
# (tag-org-source = "known") so it flows into vflowsAgg via cell 14. A known
# entry overrides a cloud label -- it's the more specific fact. Nothing is filtered.
# Missing / unreadable file -> warn and carry on with nothing known.

KNOWN_FILE = "known_ips.txt"

_rows = []
try:
    with open(KNOWN_FILE) as _f:
        for _ln in _f:
            _ln = _ln.split("#", 1)[0].split(None, 1)
            if _ln:
                _rows.append({"ip_prefix": _ln[0] if "/" in _ln[0] else _ln[0] + "/32",   # bare IP = /32
                              "label": _ln[1].strip() if len(_ln) > 1 else "known"})
except OSError as _e:
    print(f"!! {KNOWN_FILE} not loaded ({type(_e).__name__}: {_e}) -> nothing known")
known = pd.DataFrame(_rows, columns=["ip_prefix", "label"])

public["known"]       = False
public["known_label"] = pd.Series(pd.NA, index=public.index, dtype="string")
if len(known):
    _ks, _ke = cidr_bounds(known["ip_prefix"])
    _ip = public["ip_int"].to_numpy("uint64")
    for _s, _e, _l in zip(_ks, _ke, known["label"]):
        _hit = (_ip >= _s) & (_ip <= _e)
        public.loc[_hit, "known"]       = True
        public.loc[_hit, "known_label"] = _l

public.loc[public["known"], "tag-org"]      = public.loc[public["known"], "known_label"]
public.loc[public["known"], "tag-org-source"] = "known"

print(f"{len(known)} known entries -> {int(public['known'].sum())} public IPs tagged")

In [ ]:
# --- cell 14 ---
# Org / cloud-service name per session in vflowsAgg -> `tag-org`. Each public IP is looked up in both
# srcaddr and dstaddr; whichever end matches gives `tag-org` from `public` (cells 12-13).
# Only one end should be public -- sessions where both are get counted below.

_lbl   = public.set_index("ip")["tag-org"]
_src_m = vflowsAgg["srcaddr"].isin(_lbl.index)
_dst_m = vflowsAgg["dstaddr"].isin(_lbl.index)

vflowsAgg["tag-org"] = (vflowsAgg["srcaddr"].where(_src_m).map(_lbl)
                      .combine_first(vflowsAgg["dstaddr"].where(_dst_m).map(_lbl))
                      .astype("string"))

_both = int((_src_m & _dst_m).sum())
if _both:
    print(f"!! {_both:,} sessions have a public IP at BOTH ends -- tag-org taken from srcaddr")
print(f"{int((_src_m | _dst_m).sum()):,} of {len(vflowsAgg):,} sessions matched a public IP "
      f"(src {int(_src_m.sum()):,}, dst {int(_dst_m.sum()):,}); "
      f"{int(vflowsAgg['tag-org'].notna().sum()):,} of those have a tag-org")
print(vflowsAgg["tag-org"].value_counts(dropna=False).head(15).to_string())

In [ ]:
# --- cell 14a ---
# Service name per session in vflowsAgg -> `tag-service`, from portServices (cell 05c),
# matched on (protocol, port) at both ends:
#   one end's port listed            -> that service
#   both listed, same service        -> that service (e.g. NTP 123 <-> 123)
#   both listed, DIFFERENT services  -> "anomalous" (scanners sending from a fixed
#                                       well-known port, e.g. 443 -> 22)
#   neither listed                   -> <NA>
# Sessions that are neither TCP (6) nor UDP (17) -- ICMP, GRE, tunnels, ... -- get the
# protocol's name from protocolNames (conf/protocols.md) instead; <NA> if it isn't listed.

_svc   = portServices.set_index(["protocol", "port"])["service"]
_byPort = lambda col: pd.Series(pd.MultiIndex.from_arrays([vflowsAgg["protocol"], vflowsAgg[col]]).map(_svc),
                                index=vflowsAgg.index, dtype="string")
_srcSvc, _dstSvc = _byPort("srcport"), _byPort("dstport")

vflowsAgg["tag-service"] = _dstSvc.fillna(_srcSvc)
vflowsAgg.loc[(_srcSvc.notna() & _dstSvc.notna() & _srcSvc.ne(_dstSvc)).fillna(False), "tag-service"] = "anomalous"

_otherProto = vflowsAgg["protocol"].notna() & ~vflowsAgg["protocol"].isin([6, 17])
vflowsAgg.loc[_otherProto, "tag-service"] = vflowsAgg.loc[_otherProto, "protocol"].map(protocolNames).astype("string")
_unnamed = sorted(vflowsAgg.loc[_otherProto & vflowsAgg["tag-service"].isna(), "protocol"].unique().tolist())
if _unnamed:
    print(f"!! protocols not in conf/protocols.md (left <NA>): {_unnamed}")

_anom = vflowsAgg["tag-service"].eq("anomalous").fillna(False)
print(f"{int(vflowsAgg['tag-service'].notna().sum()):,} of {len(vflowsAgg):,} sessions named; "
      f"{int(_anom.sum()):,} anomalous ({int((_anom & vflowsAgg['action'].eq('ACCEPT').fillna(False)).sum()):,} ACCEPTed)")
print(vflowsAgg["tag-service"].value_counts(dropna=False).head(15).to_string())

In [ ]:
# --- cell 15 ---
# Distinct reference prefixes that matched at least one public IP

_hit_rows   = np.unique(public.loc[public["in_ref"], "ref_row"].to_numpy())
refMatches = ref_idx["rows"].iloc[_hit_rows].reset_index(drop=True)
print(refMatches["ref"].value_counts().to_string())
refMatches

In [ ]:
# --- cell 16 ---
# tag-org / tag-org-source / aws_region are set by the combined-match cell above.
# (Precedence is most-specific-prefix-wins; AWS/Azure/Zscaler address space is
#  effectively disjoint, so tag-org-source is unambiguous in practice.)

print(public["tag-org-source"].value_counts(dropna=False).to_string())
print()
print(public["tag-org"].value_counts(dropna=False).head(20).to_string())
public[["ip", "tag-org", "tag-org-source", "aws_region"]].dropna(subset=["tag-org"]).head()

In [ ]:
# --- cell 16a ---
# Column order for the session tables in cells 17 and 18. Reorder lines to move a
# column; comment one out to hide it. Derived tag-* fields first, then the flow.
# "hours" and "mb" are computed by the table cells; every other name must be a
# vflowsAgg column. A name vflowsAgg doesn't have (e.g. flow-direction on v2/v3 logs,
# or tag-label before cell 19 has run) is skipped quietly by the table cells.

TABLE_COLS = [
    "tag-direction",     # north-south / east-west
    "tag-org",           # cloud provider / known_ips.txt name
    "tag-service",       # port / protocol name, or "anomalous"
    "tag-label",         # behaviour tags, e.g. beaconing (cell 19)
    "start_time",
    "end_time",
    "hours",             # computed: end - start
    "srcaddr",
    "srcport",
    "dstaddr",
    "dstport",
    "protocol",
    "action",
    "flow-direction",    # v5 logs only
    "mb",                # computed: bytes / 1e6
    "packets",
    "records",
]

_TABLE_COMPUTED = ("hours", "mb")
_dupCols = sorted({c for c in TABLE_COLS if TABLE_COLS.count(c) > 1})
if _dupCols:
    print(f"!! listed more than once: {_dupCols}")
_later   = {"tag-label", "flow-direction"}            # may legitimately be absent here
_unknown = [c for c in TABLE_COLS if c not in vflowsAgg.columns and c not in _TABLE_COMPUTED and c not in _later]
if _unknown:
    print(f"!! not vflowsAgg columns (typo?), will be skipped: {_unknown}")
print(f"{len(TABLE_COLS)} table columns: {', '.join(TABLE_COLS)}")

In [ ]:
# --- cell 17 ---
# Top sessions in vflowsAgg by bytes or by duration. Filter by tag-direction, flow-direction, action, and whether
# the session has a tag-org. The table is an HTML widget whose .value is reassigned
# on every change (no Output widget, so nothing accumulates).

_CSS  = "font:13px system-ui,sans-serif;margin:6px 0"
# explicit cell padding: JupyterLab only pads its own DataFrame output, not HTML in widgets
_TBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
            ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
            ".flowTbl thead th{border-bottom:1px solid #888}"
            ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_wsty = {"description_width": "70px"}
_typW = widgets.ToggleButtons(options=["all", "north-south", "east-west"], description="traffic", style=_wsty)
_dirW = widgets.ToggleButtons(options=["all", "egress", "ingress"], description="direction", style=_wsty,
                                 disabled="flow-direction" not in vflowsAgg.columns)   # v2/v3 logs: no field
_actW = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], description="action", style=_wsty)
_lblW = widgets.ToggleButtons(options=["all", "org", "NA"], description="org", style=_wsty)
_srtW = widgets.ToggleButtons(options=["bytes", "duration"], description="sort by", style=_wsty)
_nW   = widgets.Dropdown(options=[25, 50, 100, 250], value=50, description="rows", style=_wsty,
                         layout=widgets.Layout(width="160px"))
_topTbl = widgets.HTML()

_topCols = TABLE_COLS          # column order: cell 16a


def _topUpdate(_=None):
    m = pd.Series(True, index=vflowsAgg.index)
    if _typW.value != "all":
        m &= vflowsAgg["tag-direction"].eq(_typW.value).fillna(False)
    if _dirW.value != "all":
        m &= vflowsAgg["flow-direction"].eq(_dirW.value).fillna(False)
    if _actW.value != "all":
        m &= vflowsAgg["action"].eq(_actW.value)
    if _lblW.value == "org":
        m &= vflowsAgg["tag-org"].notna()
    elif _lblW.value == "NA":
        m &= vflowsAgg["tag-org"].isna()

    sel = vflowsAgg[m]
    key = "bytes" if _srtW.value == "bytes" else "_dur"
    top = (sel.assign(_dur=sel["end"] - sel["start"]).dropna(subset=[key])
           .nlargest(_nW.value, key)
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: (d["_dur"] / 3600).round(2))
           [[c for c in _topCols if c in vflowsAgg.columns or c in ("mb", "hours")]])
    _topTbl.value = (
        _TBL_CSS +
        f"<div style='{_CSS}'><b>{len(sel):,}</b> sessions, "
        f"<b>{sel['bytes'].sum() / 1e6:,.1f} MB</b> match &mdash; top {len(top)} by {_srtW.value}</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


for _w in (_typW, _dirW, _actW, _lblW, _srtW, _nW):
    _w.observe(_topUpdate, names="value")

_topUpdate()
display(widgets.VBox([_typW, _dirW, _actW, _lblW, widgets.HBox([_srtW, _nW]), _topTbl]))

In [ ]:
# --- cell 18 ---
# Longest-running sessions in vflowsAgg (same filters as the top-bytes cell above,
# ranked by duration = end - start, and the summary sums hours instead of MB).
# Own widget names, so it doesn't share state with the cell above.

_CSS  = "font:13px system-ui,sans-serif;margin:6px 0"
# explicit cell padding: JupyterLab only pads its own DataFrame output, not HTML in widgets
_TBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
            ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
            ".flowTbl thead th{border-bottom:1px solid #888}"
            ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_wsty = {"description_width": "70px"}
_durTypW = widgets.ToggleButtons(options=["all", "north-south", "east-west"], description="traffic", style=_wsty)
_durDirW = widgets.ToggleButtons(options=["all", "egress", "ingress"], description="direction", style=_wsty,
                                    disabled="flow-direction" not in vflowsAgg.columns)   # v2/v3 logs: no field
_durActW = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], description="action", style=_wsty)
_durLblW = widgets.ToggleButtons(options=["all", "org", "NA"], description="org", style=_wsty)
_durNW   = widgets.Dropdown(options=[25, 50, 100, 250], value=50, description="rows", style=_wsty,
                            layout=widgets.Layout(width="160px"))
_durTbl  = widgets.HTML()

_durCols = TABLE_COLS          # column order: cell 16a


def _durUpdate(_=None):
    m = pd.Series(True, index=vflowsAgg.index)
    if _durTypW.value != "all":
        m &= vflowsAgg["tag-direction"].eq(_durTypW.value).fillna(False)
    if _durDirW.value != "all":
        m &= vflowsAgg["flow-direction"].eq(_durDirW.value).fillna(False)
    if _durActW.value != "all":
        m &= vflowsAgg["action"].eq(_durActW.value)
    if _durLblW.value == "org":
        m &= vflowsAgg["tag-org"].notna()
    elif _durLblW.value == "NA":
        m &= vflowsAgg["tag-org"].isna()

    sel = vflowsAgg[m].assign(hours=lambda d: (d["end"] - d["start"]) / 3600)
    top = (sel.dropna(subset=["hours"]).nlargest(_durNW.value, "hours")
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: d["hours"].round(2))
           [[c for c in _durCols if c in vflowsAgg.columns or c in ("mb", "hours")]])
    _durTbl.value = (
        _TBL_CSS +
        f"<div style='{_CSS}'><b>{len(sel):,}</b> sessions, "
        f"<b>{sel['hours'].sum():,.1f} hours</b> match &mdash; top {len(top)} by duration</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


for _w in (_durTypW, _durDirW, _durActW, _durLblW, _durNW):
    _w.observe(_durUpdate, names="value")

_durUpdate()
display(widgets.VBox([_durTypW, _durDirW, _durActW, widgets.HBox([_durLblW, _durNW]), _durTbl]))

In [ ]:
# --- cell 19 ---
# Beacon candidates: outbound, allowed traffic that checks in on a regular interval
# with small, steady payloads over a long period. Groups RAW records (vflows, not
# vflowsAgg -- merging loses the per-record timing) by local host -> remote addr,
# port, protocol, ignoring source port, so it catches both a held-open connection
# (one srcport) and one that reconnects every check-in (a new srcport each time).
# Caveat: flow logs bucket traffic into ~1-minute capture windows, so any steady
# connection produces ~60 s records -- the small-payload limits are what separate
# a beacon from a busy stream. Intervals much under a minute can't be resolved.
# Candidates get tag-label = "beaconing", here and on their ACCEPTed outbound sessions
# in vflowsAgg (same srcaddr, dstaddr, dstport, protocol), so cell 20 can query them.
# Re-running clears the previous run's "beaconing" tags first.

BEACON_INTERVAL_S = (25, 75)   # median check-in interval must fall in this range ("30-60 s, give or take")
BEACON_JITTER     = 0.25       # an interval counts as on-beat if within +/-25% of the median
BEACON_REGULAR    = 0.60       # ...and at least this share of intervals must be on-beat
BEACON_MAX_BYTES  = 2000       # median bytes per record (outbound) at most
BEACON_MAX_PKTS   = 20         # median packets per record (outbound) at most
BEACON_MIN_HOURS  = 6          # first-to-last span at least
BEACON_MIN_RECS   = 60         # outbound records at least

_pubIps = set(public["ip"])
_bk     = ["srcaddr", "dstaddr", "dstport", "protocol"]
_out    = vflows[vflows["action"].eq("ACCEPT") & vflows["dstaddr"].isin(_pubIps) & ~vflows["srcaddr"].isin(_pubIps)]
_back   = vflows[vflows["action"].eq("ACCEPT") & vflows["srcaddr"].isin(_pubIps) & ~vflows["dstaddr"].isin(_pubIps)]

# check-in intervals: gaps between distinct record start times per group
_t = _out[_bk + ["start"]].drop_duplicates().sort_values(_bk + ["start"])
_t["gap"] = _t.groupby(_bk)["start"].diff()
_t = _t.dropna(subset=["gap"])
_t["onBeat"] = ((_t["gap"] - _t.groupby(_bk)["gap"].transform("median")).abs()
                <= BEACON_JITTER * _t.groupby(_bk)["gap"].transform("median"))
_iv = _t.groupby(_bk).agg(interval_s=("gap", "median"), regular=("onBeat", "mean"))

_st = _out.groupby(_bk).agg(records=("bytes", "size"), srcports=("srcport", "nunique"),
                            first=("start_time", "min"), last=("end_time", "max"),
                            _s=("start", "min"), _e=("end", "max"),
                            bytesRec=("bytes", "median"), pktsRec=("packets", "median"),
                            bytesOut=("bytes", "sum"))
_in = (_back[["dstaddr", "srcaddr", "srcport", "protocol", "bytes"]]           # replies, flipped to the outbound key
       .set_axis(_bk + ["bytes"], axis=1)
       .groupby(_bk)["bytes"].sum().rename("bytesIn"))

beaconCandidates = (_st.join(_iv).join(_in)
    .assign(spanHours=lambda d: ((d["_e"] - d["_s"]) / 3600).round(1),
            kbOut=lambda d: (d["bytesOut"] / 1e3).round(1),
            kbIn=lambda d: (d["bytesIn"] / 1e3).round(1),
            regular=lambda d: d["regular"].astype(float).round(2))
    .drop(columns=["_s", "_e"])
    .reset_index())
beaconCandidates["tag-org"] = beaconCandidates["dstaddr"].map(public.set_index("ip")["tag-org"]).astype("string")

_lo, _hi = BEACON_INTERVAL_S
_hit = (beaconCandidates["interval_s"].between(_lo, _hi)
        & beaconCandidates["regular"].ge(BEACON_REGULAR)
        & beaconCandidates["bytesRec"].le(BEACON_MAX_BYTES)
        & beaconCandidates["pktsRec"].le(BEACON_MAX_PKTS)
        & beaconCandidates["spanHours"].ge(BEACON_MIN_HOURS)
        & beaconCandidates["records"].ge(BEACON_MIN_RECS))
beaconCandidates["tag-label"] = "beaconing"
beaconCandidates = (beaconCandidates[_hit]
    [["srcaddr", "dstaddr", "dstport", "protocol", "tag-org", "tag-label", "interval_s", "regular",
      "bytesRec", "pktsRec", "records", "srcports", "spanHours", "first", "last", "kbOut", "kbIn"]]
    .sort_values("spanHours", ascending=False)
    .reset_index(drop=True))

print(f"interval {_lo}-{_hi} s, >= {BEACON_REGULAR:.0%} on-beat (+/-{BEACON_JITTER:.0%}), "
      f"<= {BEACON_MAX_BYTES} B and <= {BEACON_MAX_PKTS} pkts per record, "
      f">= {BEACON_MIN_HOURS} h, >= {BEACON_MIN_RECS} records")
if "tag-label" not in vflowsAgg.columns:
    vflowsAgg["tag-label"] = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
vflowsAgg.loc[vflowsAgg["tag-label"].eq("beaconing").fillna(False), "tag-label"] = pd.NA
_bSess = (pd.MultiIndex.from_frame(vflowsAgg[_bk]).isin(pd.MultiIndex.from_frame(beaconCandidates[_bk]))
          & vflowsAgg["action"].eq("ACCEPT").fillna(False).to_numpy())
vflowsAgg.loc[_bSess, "tag-label"] = "beaconing"

print(f"{len(beaconCandidates)} beacon candidates of {len(_st):,} outbound host/peer/port groups")
print(f"{int(_bSess.sum()):,} vflowsAgg sessions tagged tag-label = beaconing")
beaconCandidates

In [ ]:
# --- cell 20 ---
# Query vflowsAgg on any combination of srcaddr, srcport, dstaddr, dstport, protocol,
# tag-direction, action, flow-direction, tag-org, tag-service and tag-label. Blank / "all" = no filter; filled-in fields AND
# together. Text fields take comma-separated terms that OR together, and a leading
# ! excludes a term:
#   addresses  10.0.0.5, 10.0.0.0/16, !10.0.0.9     (IPv4 / CIDR; IPv6 exact match)
#   ports      443, 8000-8999, !22
#   protocol   tcp, udp, icmp or numbers -- 6, 17, !1
#   org        substring, any case -- amazon, !known; NA = no tag-org
#   service    tag-service, same rules -- ssh, rdp, anomalous; NA = port not in conf/
#   label      tag-label, same rules -- beaconing; NA = untagged (set by cell 19)
# "either addr" / "either port" match a term at src OR dst; a ! term there drops
# sessions with that value at either end (e.g. !10.0.0.9 = doesn't involve 10.0.0.9).
# Text fields apply on Enter or when you click away. The full match (not just the rows
# shown) is left in `queryResult` for further pandas work.

import html

_qCSS = "font:13px system-ui,sans-serif;margin:6px 0"
_qTBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
             ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
             ".flowTbl thead th{border-bottom:1px solid #888}"
             ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_qSty  = {"description_width": "80px"}
_qCols = ["start_time", "end_time", "hours", "srcaddr", "srcport", "dstaddr", "dstport",
          "protocol", "tag-direction", "action", "flow-direction", "tag-org", "tag-service", "tag-label", "mb", "packets", "records"]
_qProto = {"icmp": 1, "tcp": 6, "udp": 17, "gre": 47, "esp": 50, "icmpv6": 58}


def _qIpInts(col):
    """Per row: IPv4 address as int, or a sentinel above the IPv4 range (None / IPv6)."""
    s   = vflowsAgg[col].astype("string")
    ok  = s.str.count(r"\.").eq(3).fillna(False).to_numpy()
    out = np.full(len(s), np.uint64(1 << 40))
    u, inv = np.unique(s[ok].to_numpy(dtype=object), return_inverse=True)
    if len(u):
        out[ok] = ipv4_to_int(u)[inv]
    return out

_qInts = {"srcaddr": _qIpInts("srcaddr"), "dstaddr": _qIpInts("dstaddr")}   # parsed once


def _qTerms(txt):
    """'a, !b, c' -> (['a', 'c'], ['b'])"""
    inc, exc = [], []
    for t in txt.replace(" ", ",").split(","):
        if t.strip("!"):
            (exc if t.startswith("!") else inc).append(t.lstrip("!"))
    return inc, exc


def _qApply(m, txt, termMask):
    """AND the OR of the included terms into m, then drop every excluded term."""
    inc, exc = _qTerms(txt)
    if inc:
        m &= reduce(np.logical_or, (termMask(t) for t in inc))
    for t in exc:
        m &= ~termMask(t)
    return m


def _qAddrMask(col, name=None):
    def f(t):
        if ":" in t:                                               # IPv6: exact match
            return vflowsAgg[col].eq(t).fillna(False).to_numpy()
        try:
            ipaddress.IPv4Network(t, strict=False)
        except ValueError:
            raise ValueError(f"{name or col}: {t!r} is not an IPv4 address or CIDR") from None
        lo, hi = cidr_bounds([t if "/" in t else t + "/32"])
        return (_qInts[col] >= lo[0]) & (_qInts[col] <= hi[0])
    return f


def _qNumMask(col, names=None, name=None):
    def f(t):
        t = (names or {}).get(t.lower(), t)
        try:
            lo, _, hi = str(t).partition("-")
            lo, hi = int(lo), int(hi or lo)
        except ValueError:
            raise ValueError(f"{name or col}: {t!r} is not a number or range"
                             + (f" (or one of {', '.join(names)})" if names else "")) from None
        return vflowsAgg[col].between(lo, hi).fillna(False).to_numpy()
    return f


def _qEither(maskA, maskB):
    """Term matches if it matches either end."""
    return lambda t: maskA(t) | maskB(t)


def _qTagMask(col):
    """Substring match, any case; NA = empty. A column that doesn't exist matches nothing."""
    def f(t):
        if col not in vflowsAgg.columns:
            return np.full(len(vflowsAgg), t.upper() == "NA")
        if t.upper() == "NA":
            return vflowsAgg[col].isna().to_numpy()
        return vflowsAgg[col].astype("string").str.contains(t, case=False, regex=False).fillna(False).to_numpy()
    return f


def _qText(desc, ph):
    return widgets.Text(description=desc, placeholder=ph, continuous_update=False, style=_qSty,
                        layout=widgets.Layout(width="420px"))

_qSrcW   = _qText("srcaddr",  "10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qSportW = _qText("srcport",  "443, 8000-8999, !22")
_qDstW   = _qText("dstaddr",  "10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qDportW = _qText("dstport",  "443, 8000-8999, !22")
_qAnyAW  = _qText("either addr", "src OR dst: 10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qAnyPW  = _qText("either port", "src OR dst: 443, 8000-8999, !22")
_qProtW  = _qText("protocol", "tcp, udp, icmp, 6, !17")
_qLblW   = _qText("org",      "amazon, !known, NA")
_qSvcW   = _qText("service",  "ssh, rdp, anomalous, NA")
_qTagW   = _qText("label",    "beaconing, NA")
_qTypW   = widgets.ToggleButtons(options=["all", "north-south", "east-west", "NA"], description="traffic", style=_qSty)
_qActW   = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT", "NA"], description="action", style=_qSty)
_qDirW   = widgets.ToggleButtons(options=["all", "egress", "ingress", "NA"], description="direction", style=_qSty,
                                 disabled="flow-direction" not in vflowsAgg.columns)   # v2/v3 logs: no field
_qSrtW   = widgets.Dropdown(options=["bytes", "duration", "packets", "records", "latest", "earliest"],
                            description="sort by", style=_qSty, layout=widgets.Layout(width="220px"))
_qNW     = widgets.Dropdown(options=[25, 50, 100, 250, 1000], value=50, description="rows", style=_qSty,
                            layout=widgets.Layout(width="160px"))
_qClrB   = widgets.Button(description="clear filters", layout=widgets.Layout(width="120px"))
_qTbl    = widgets.HTML()
_qTextWs = (_qSrcW, _qSportW, _qDstW, _qDportW, _qAnyAW, _qAnyPW, _qProtW, _qLblW, _qSvcW, _qTagW)
queryResult = vflowsAgg.iloc[:0]


def _qUpdate(_=None):
    global queryResult
    try:
        m = np.ones(len(vflowsAgg), dtype=bool)
        m = _qApply(m, _qSrcW.value,   _qAddrMask("srcaddr"))
        m = _qApply(m, _qDstW.value,   _qAddrMask("dstaddr"))
        m = _qApply(m, _qSportW.value, _qNumMask("srcport"))
        m = _qApply(m, _qDportW.value, _qNumMask("dstport"))
        m = _qApply(m, _qAnyAW.value,  _qEither(_qAddrMask("srcaddr", "either addr"),
                                               _qAddrMask("dstaddr", "either addr")))
        m = _qApply(m, _qAnyPW.value,  _qEither(_qNumMask("srcport", name="either port"),
                                               _qNumMask("dstport", name="either port")))
        m = _qApply(m, _qProtW.value,  _qNumMask("protocol", _qProto))
        m = _qApply(m, _qLblW.value,   _qTagMask("tag-org"))
        m = _qApply(m, _qSvcW.value,   _qTagMask("tag-service"))
        m = _qApply(m, _qTagW.value,   _qTagMask("tag-label"))
    except ValueError as e:
        _qTbl.value = f"<div style='{_qCSS};color:#c62828'><b>!!</b> {html.escape(str(e))}</div>"
        return
    for w, col in ((_qTypW, "tag-direction"), (_qActW, "action"), (_qDirW, "flow-direction")):
        if col not in vflowsAgg.columns:
            continue
        if w.value == "NA":
            m &= vflowsAgg[col].isna().to_numpy()
        elif w.value != "all":
            m &= vflowsAgg[col].eq(w.value).fillna(False).to_numpy()

    queryResult = vflowsAgg[m].assign(_dur=lambda d: d["end"] - d["start"])
    key, asc = {"duration": ("_dur", False), "latest": ("start", False),
                "earliest": ("start", True)}.get(_qSrtW.value, (_qSrtW.value, False))
    top = (queryResult.sort_values(key, ascending=asc, na_position="last", kind="stable")
           .head(_qNW.value)
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: (d["_dur"] / 3600).round(2))
           [[c for c in _qCols if c in vflowsAgg.columns or c in ("mb", "hours")]])
    queryResult = queryResult.drop(columns="_dur")
    _qTbl.value = (
        _qTBL_CSS +
        f"<div style='{_qCSS}'><b>{len(queryResult):,}</b> sessions, "
        f"<b>{queryResult['bytes'].sum() / 1e6:,.1f} MB</b>, "
        f"<b>{queryResult['records'].sum():,}</b> records match &mdash; "
        f"showing {len(top)} by {_qSrtW.value} (all matches in <code>queryResult</code>)</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


def _qClear(_=None):
    for w in _qTextWs + (_qTypW, _qActW, _qDirW):
        w.unobserve(_qUpdate, names="value")
        w.value = "all" if isinstance(w, widgets.ToggleButtons) else ""
        w.observe(_qUpdate, names="value")
    _qUpdate()


for _w in _qTextWs + (_qTypW, _qActW, _qDirW, _qSrtW, _qNW):
    _w.observe(_qUpdate, names="value")
_qClrB.on_click(_qClear)

_qUpdate()
display(widgets.VBox([
    widgets.HBox([_qSrcW, _qSportW]),
    widgets.HBox([_qDstW, _qDportW]),
    widgets.HBox([_qAnyAW, _qAnyPW]),
    widgets.HBox([_qProtW, _qLblW]),
    widgets.HBox([_qSvcW, _qTagW]),
    _qTypW, _qActW, _qDirW,
    widgets.HBox([_qSrtW, _qNW, _qClrB]),
    _qTbl]))

In [ ]:
# --- cell 21 ---
# Sessions and bytes by tag-service (vflowsAgg). Two panels on one shared service axis
# -- never one chart with two scales. Sorted by sessions. Colors follow the service, not
# its rank: the 5 busiest named services (over all of vflowsAgg, fixed when this cell
# runs) get their own color and keep it through every filter; other named services are
# light gray; red = "anomalous" (src and dst ports name different services); mid gray =
# port not in conf/. Every bar is also labeled, so color is never the only cue.
# Filters redraw the chart (a plain figure, not a FigureWidget -- see _svcFigure); the table under the chart holds the same numbers.
# The chart shows the top N services by sessions (SVC_TOP_N, "show" dropdown); the table
# and `serviceSummary` always hold every service.

import html
from plotly.subplots import make_subplots

# validated (dataviz validator, all pairs incl. the anomalous red): blue, yellow, magenta,
# violet, aqua. Orange and green are left out -- too close to the anomalous red.
_SVC_PALETTE = ["#2a78d6", "#eda100", "#e87ba4", "#4a3aa7", "#1baf7a"]
_SVC_OTHER   = "#c3c2b7"                                                  # remaining named services
_SVC_COLORS  = {"anomalous": "#d03b3b", "(unlisted port)": "#898781"}    # status-critical, muted gray
_svcRank = (vflowsAgg["tag-service"].dropna().loc[lambda s: s.ne("anomalous")]
            .value_counts().index[:len(_SVC_PALETTE)])
_SVC_COLORS.update(dict(zip(_svcRank, _SVC_PALETTE)))
_INK, _INK2, _GRID, _SURF = "#0b0b0b", "#52514e", "#e8e7e2", "#fcfcfb"
SVC_TOP_N = 10


def _fmtBytes(b):
    for u in ("B", "kB", "MB", "GB", "TB"):
        if abs(b) < 1000 or u == "TB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


def _svcFigure(names, colors, sessions, nbytes, cdata, log, title):
    """A plain (one-way) plotly figure. Drawn fresh on every change -- no FigureWidget,
    whose two-way sync with the VS Code front end can hang the notebook."""
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.10,
                        subplot_titles=("Sessions", "Bytes"))
    for col, vals, txt in ((1, sessions, [f"{v:,}" for v in sessions]),
                           (2, nbytes,   [_fmtBytes(v) for v in nbytes])):
        fig.add_bar(row=1, col=col, orientation="h", y=names, x=vals, text=txt, customdata=cdata,
                    marker=dict(color=colors, cornerradius=4), textposition="outside", cliponaxis=False,
                    textfont_color=_INK2, showlegend=False,
                    hovertemplate="<b>%{y}</b><br>%{customdata[0]:,} sessions (%{customdata[1]:.1%})"
                                  "<br>%{customdata[2]} (%{customdata[3]:.1%})"
                                  "<br>%{customdata[4]:,} raw records<extra></extra>")
    # ~26 px per service row; bars capped at ~22 px thick however few rows there are
    h = max(180, 26 * len(names) + 90)
    fig.update_layout(height=h, bargap=max(0.35, 1 - 22 / ((h - 90) / max(len(names), 1))),
                      paper_bgcolor=_SURF, plot_bgcolor=_SURF, margin=dict(l=20, r=70, t=60, b=30),
                      font=dict(family="system-ui, sans-serif", size=12, color=_INK),
                      hoverlabel=dict(bgcolor=_SURF, font_color=_INK),
                      title=dict(text=title, x=0.01, xref="container", xanchor="left", font=dict(size=15, color=_INK)))
    # SI ticks; bytes get a "B" suffix so 1G reads 1 GB, not plotly's "1B" (= billion)
    fig.update_xaxes(type="log" if log else "linear", tickformat="~s", dtick=1 if log else None,
                     gridcolor=_GRID, gridwidth=1, zeroline=False, tickfont_color=_INK2, rangemode="tozero")
    fig.update_xaxes(row=1, col=2, ticksuffix="B")
    fig.update_yaxes(categoryorder="array", categoryarray=names[::-1], tickfont_color=_INK, ticksuffix="  ",
                     showgrid=False)
    fig.update_annotations(font=dict(size=13, color=_INK))
    return fig


_svcActW   = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], description="action", style=_wsty)
_svcTypW   = widgets.ToggleButtons(options=["all", "north-south", "east-west"], description="traffic", style=_wsty)
_svcScaleW = widgets.ToggleButtons(options=["linear", "log"], description="x scale", style=_wsty)
_svcTopW   = widgets.Dropdown(options=[("top 10", 10), ("top 20", 20), ("all", 0)], value=SVC_TOP_N,
                              description="show", style=_wsty, layout=widgets.Layout(width="180px"))
_svcOut    = widgets.Output()          # cleared and redrawn on each change, so nothing piles up
_svcTbl    = widgets.HTML()
serviceSummary = None


def _svcUpdate(_=None):
    global serviceSummary
    sel = vflowsAgg
    if _svcActW.value != "all":
        sel = sel[sel["action"].eq(_svcActW.value).fillna(False)]
    if _svcTypW.value != "all":
        sel = sel[sel["tag-direction"].eq(_svcTypW.value).fillna(False)]

    s = (sel.assign(service=sel["tag-service"].astype("string").fillna("(unlisted port)"))
         .groupby("service")
         .agg(sessions=("records", "size"), bytes=("bytes", "sum"), records=("records", "sum"))
         .sort_values(["sessions", "bytes"], ascending=False))
    s["bytes"] = s["bytes"].fillna(0).astype("int64")
    s["sessionShare"] = s["sessions"] / max(int(s["sessions"].sum()), 1)
    s["byteShare"]    = s["bytes"] / max(int(s["bytes"].sum()), 1)
    serviceSummary = s

    top    = s.head(_svcTopW.value) if _svcTopW.value else s
    names  = top.index.tolist()
    colors = [_SVC_COLORS.get(n, _SVC_OTHER) for n in names]
    cdata  = list(zip(top["sessions"], top["sessionShare"], top["bytes"].map(_fmtBytes), top["byteShare"], top["records"]))
    shown  = f"top {len(top)} of {len(s)} services" if len(top) < len(s) else f"all {len(s)} services"
    title  = (f"Sessions and bytes by tag-service &nbsp;<span style='font-size:12px;color:{_INK2}'>"
              f"{shown}  ·  {len(sel):,} sessions, {_fmtBytes(int(sel['bytes'].sum()))}</span>")
    fig    = _svcFigure(names, colors, top["sessions"].tolist(), top["bytes"].tolist(), cdata,
                        _svcScaleW.value == "log", title)
    with _svcOut:
        _svcOut.clear_output(wait=True)
        display(fig)

    tbl = s.assign(bytes=s["bytes"].map(_fmtBytes), sessionShare=s["sessionShare"].map("{:.1%}".format),
                   byteShare=s["byteShare"].map("{:.1%}".format), sessions=s["sessions"].map("{:,}".format),
                   records=s["records"].map("{:,}".format))
    _svcTbl.value = (_TBL_CSS + "<details><summary style='font:13px system-ui,sans-serif;cursor:pointer'>table view</summary>"
                     + tbl.reset_index().to_html(index=False, border=0, escape=True).replace("<table", "<table class='flowTbl'")
                     + "</details>")


def _svcSafeUpdate(_=None):
    """Errors inside widget callbacks are swallowed by the front end -- show them instead."""
    try:
        _svcUpdate()
    except Exception as e:
        _svcTbl.value = (f"<div style='font:13px system-ui,sans-serif;color:#c62828'><b>!! chart update failed:</b> "
                         f"{html.escape(f'{type(e).__name__}: {e}')}</div>")


for _w in (_svcActW, _svcTypW, _svcScaleW, _svcTopW):
    _w.observe(_svcSafeUpdate, names="value")

_svcUpdate()
display(widgets.VBox([_svcActW, _svcTypW, widgets.HBox([_svcScaleW, _svcTopW]), _svcOut, _svcTbl]))